# ARC2 018-A frozen F-291 feedback vs restart
2 model calls, 12288 each. No human patch in feedback. 45min cap.


In [ ]:
import base64, hashlib, json, os, signal, subprocess, sys, time
from pathlib import Path
code_dir = Path('/kaggle/working/018-aq-code')
code_dir.mkdir(exist_ok=False)
frozen_dir = Path('/kaggle/working/018-frozen')
frozen_dir.mkdir(exist_ok=False)
blobs = json.loads(base64.b64decode('...'))
expected = {"vdriver018a.py": "9292c23de710147f3484b090097b19d1935e42b6f0fedd7dd3bbbc56e922ebd3", "run_reasoning_probe.py": "7aac22b620bee0cda3b7d930b31e641b6c02a674d981421a60ee48863838a0d5", "solve_program.py": "03aaa3818a8f3beafee019140bae823f968e913af72ef297fa7531f24816a37a", "F-291dc1e1_0.final.txt": "16e3c18ea494b1f866c54136db4bde6a91ad6a071f59e84ddc63af4d42f709d1", "F-291dc1e1_0.code.py": "cc808fdb875a395dced5601f22616e60606306e57d115812125e98a6f0a1f9de", "task-291-clean.json": "6a0c60cfb64acd658deb95fd4ba2fca7b932023584fabc921b691f04ae8ff1ef"}
layout = {'vdriver018a.py': code_dir, 'run_reasoning_probe.py': code_dir, 'solve_program.py': code_dir, 'F-291dc1e1_0.final.txt': frozen_dir, 'F-291dc1e1_0.code.py': frozen_dir, 'task-291-clean.json': frozen_dir}
for name, encoded in blobs.items():
    raw = base64.b64decode(encoded)
    assert hashlib.sha256(raw).hexdigest() == expected[name], name
    (layout[name] / name).write_bytes(raw)
PY312 = '/usr/bin/python3.12'
assert Path(PY312).is_file(), 'python3.12 binary missing'
proc = subprocess.Popen([PY312, '-u', str(code_dir / 'vdriver018a.py'), '--work', '/kaggle/working/vprobe', '--frozen', str(frozen_dir), '--session-seconds', '2700', '--pip-python', sys.executable], cwd=str(code_dir), start_new_session=True)
try:
    rc = proc.wait(timeout=2640)
except subprocess.TimeoutExpired:
    print('NOTEBOOK_DEADLINE', flush=True)
    os.killpg(proc.pid, signal.SIGTERM)
    try:
        rc = proc.wait(timeout=90)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        rc = proc.wait(timeout=30)
print('VDRIVER_RETURN_CODE', rc, flush=True)
assert rc == 0, 'Inspect /kaggle/working/vprobe/run.json'
